# TRAJ DATA

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, DeepGraphInfomax

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score, mean_squared_error
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

In [2]:
locomizer_OD = pd.read_csv('../data/msoa_OD_allactivity.csv.gz')
locomizer_OD

,o_msoa,d_msoa,num_trips
0,E02007005,E02007005,28854
1,E02000524,E02000524,27280
2,E02000001,E02000001,22293
3,E02000977,E02000977,21419
4,E02007099,E02007099,21154
...,...,...,...
2502831,E02006880,E02006790,1
2502832,E02005810,E02002380,1
2502833,E02004989,E02000658,1
2502834,E02000978,E02003408,1


In [3]:
locomizer_total_od = (
    locomizer_OD
    .groupby(["o_msoa", "d_msoa"])["num_trips"]
    .sum()
    .reset_index()
)

locomizer_total_od.rename(columns={"num_trips": "flow"}, inplace=True)

print("Number of OD pairs:", len(locomizer_total_od))
locomizer_total_od.head()

Number of OD pairs: 2502265


,o_msoa,d_msoa,flow
0,E02000001,E02000001,22293
1,E02000001,E02000002,8
2,E02000001,E02000003,25
3,E02000001,E02000004,16
4,E02000001,E02000005,28


In [4]:
import geopandas as gpd
# Read POIs (gpkg) and LSOA boundaries 
poi= gpd.read_file("../data/poi_uk.gpkg")       

# Replacing CSV with conversion to GeoDataFrame (using BNG eastings/northings) ---
LSOA_df = pd.read_csv("../data/Lower_layer_Super_Output_Areas_(December_2021)_Boundaries_EW_BFE_(V10)_and_RUC.csv")


#### I need to aggregate the POI to MSOA level

In [5]:
poi.head()

,id,primary_name,main_category,alternate_category,address,locality,postcode,region,country,source,source_record_id,lat,long,h3_15,easting,northing,LSOA21CD,geometry
0,08f187290304e43003f7ace9f8da4b96,Gav's Fish and Chips,restaurant,fish_and_chips_restaurant|diner,Troytown Farm,None,TR22 0PL,None,GB,meta,104198436036821,49.891107,-6.351415,8f187290304e430,87580.859066,8070.307149,E01019077,POINT (87580.859 8070.307)
1,08f187290305a2db036e8c44b2ca7b73,Troytown Farm & Campsite,campground,farm|dairy_farm,"Troytown Farm, St Agnes",None,TR22 0PL,None,GB,meta,291582648427,49.891448,-6.351431,8f1872903043bb5,87581.918498,8108.314168,E01019077,POINT (87581.918 8108.314)
2,08f18729030624ee032a3a8a746fde6a,St Agnes Watersports,canoe_and_kayak_hire_service,boat_rental_and_training,Hellweathers,None,TR22 0PL,None,GB,meta,199148563923007,49.892475,-6.350767,8f1872903063cb1,87636.283014,8219.679675,E01019077,POINT (87636.283 8219.68)
3,08f1872903a9d234032557729d2a66bd,Coastguards Lookout,cafe,restaurant|coffee_shop,None,None,TR22 0PL,None,GB,meta,1093236587411373,49.891297,-6.346946,8f1872903a9d260,87902.952621,8072.823956,E01019077,POINT (87902.953 8072.824)
4,08f187290314b50303a6ae3f66d2ce03,St Agnes Church,church_cathedral,social_service_organizations|catholic_church,New Ln,None,TR22 0PL,None,GB,meta,132002323501511,49.893181,-6.349500,8f187290314b0e6,87731.792387,8292.790922,E01019077,POINT (87731.792 8292.791)


In [6]:
lookup_lsoa = pd.read_csv("../data/PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv") #the LSOA- MSOA lookup file

C:\Users\jzvf437\AppData\Local\Temp\ipykernel_30740\2412116378.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup_lsoa = pd.read_csv("../data/PCD_OA21_LSOA21_MSOA21_LAD_MAY25_UK_LU.csv") #the LSOA- MSOA lookup file


In [7]:
# Basic mapping from main_category to broader group
category_to_group = {

    # FOOD & DRINK
    'restaurant': 'FOOD_DRINK',
    'pub': 'FOOD_DRINK',
    'bar': 'FOOD_DRINK',
    'cafe': 'FOOD_DRINK',
    'coffee_shop': 'FOOD_DRINK',
    'fast_food': 'FOOD_DRINK',
    'fish_and_chips_restaurant': 'FOOD_DRINK',
    'chilean_restaurant': 'FOOD_DRINK',

    # SHOPPING / RETAIL
    'shopping': 'RETAIL',
    'shopping_mall': 'RETAIL',
    'grocery_or_supermarket': 'RETAIL',
    'convenience_store': 'RETAIL',
    'department_store': 'RETAIL',

    # LEISURE / SPORT
    'active_life': 'LEISURE_SPORT',
    'gym': 'LEISURE_SPORT',
    'sports_club': 'LEISURE_SPORT',
    'stadium': 'LEISURE_SPORT',
    'cinema': 'LEISURE_SPORT',
    'museum': 'LEISURE_SPORT',
    'tourist_attraction': 'LEISURE_SPORT',

    # BEAUTY & PERSONAL
    'beauty_and_spa': 'BEAUTY_PERSONAL',
    'hair_salon': 'BEAUTY_PERSONAL',
    'nail_salon': 'BEAUTY_PERSONAL',
    'spa': 'BEAUTY_PERSONAL',

    # PROFESSIONAL / BUSINESS
    'professional_services': 'PROFESSIONAL',
    'lawyer': 'PROFESSIONAL',
    'accountant': 'PROFESSIONAL',
    'consulting_agency': 'PROFESSIONAL',

    # HEALTH
    'hospital': 'HEALTH',
    'doctor': 'HEALTH',
    'dentist': 'HEALTH',
    'pharmacy': 'HEALTH',
    'elder_care_planning': 'HEALTH',

    # EDUCATION
    'school': 'EDUCATION',
    'primary_school': 'EDUCATION',
    'secondary_school': 'EDUCATION',
    'college': 'EDUCATION',
    'university': 'EDUCATION',
    'parenting_classes': 'EDUCATION',

    # AUTO
    'tire_repair_shop': 'AUTO',
    'car_repair': 'AUTO',
    'car_dealer': 'AUTO',
    'petrol_station': 'AUTO',

    # NATURE / OUTDOOR
    'park': 'NATURE_OUTDOOR',
    'sand_dune': 'NATURE_OUTDOOR',
    'beach': 'NATURE_OUTDOOR',
    'forest': 'NATURE_OUTDOOR',

    # RELIGION
    'church_cathedral': 'RELIGION',
    'church': 'RELIGION',
    'mosque': 'RELIGION',
    'synagogue': 'RELIGION',
    'temple': 'RELIGION',

    # ACCOMMODATION
    'hotel': 'ACCOMMODATION',
    'motel': 'ACCOMMODATION',
    'campground': 'ACCOMMODATION',
    'guest_house': 'ACCOMMODATION',
    'holiday_rental_home': 'ACCOMMODATION',
}


In [8]:
# Map each POI to a broad group
poi['poi_group'] = poi['main_category'].map(category_to_group)

# Check how many got mapped
print(poi['poi_group'].value_counts(dropna=False).head(20))

# keep only mapped rows to start simple
poi_mapped = poi[~poi['poi_group'].isna()].copy()

poi_group
NaN                2003083
FOOD_DRINK          118348
PROFESSIONAL         87777
BEAUTY_PERSONAL      87072
RETAIL               61169
LEISURE_SPORT        48277
HEALTH               41098
ACCOMMODATION        31645
RELIGION             20421
AUTO                 10648
NATURE_OUTDOOR        7396
EDUCATION             4248
Name: count, dtype: int64


In [9]:
lookup_lsoa.head()

,pcd7,pcd8,pcds,dointr,doterm,usertype,oa21cd,lsoa21cd,msoa21cd,ladcd,lsoa21nm,msoa21nm,ladnm,ladnmw
0,AB1 0AA,AB1 0AA,AB1 0AA,198001,199606.0,0,S00137176,S01013490,S02002516,S12000033,"Cults, Bieldside and Milltimber West - 02","Cults, Bieldside and Milltimber West",Aberdeen City,NaN
1,AB1 0AB,AB1 0AB,AB1 0AB,198001,199606.0,0,S00137176,S01013490,S02002516,S12000033,"Cults, Bieldside and Milltimber West - 02","Cults, Bieldside and Milltimber West",Aberdeen City,NaN
2,AB1 0AD,AB1 0AD,AB1 0AD,198001,199606.0,0,S00137176,S01013490,S02002516,S12000033,"Cults, Bieldside and Milltimber West - 02","Cults, Bieldside and Milltimber West",Aberdeen City,NaN
3,AB1 0AE,AB1 0AE,AB1 0AE,199402,199606.0,0,S00138891,S01013856,S02002577,S12000034,"Dunecht, Durris and Drumoak - 01","Dunecht, Durris and Drumoak",Aberdeenshire,NaN
4,AB1 0AF,AB1 0AF,AB1 0AF,199012,199207.0,1,S00137241,S01013487,S02002515,S12000033,Culter - 06,Culter,Aberdeen City,NaN


In [10]:
lsoa_msoa = lookup_lsoa[['lsoa21cd', 'msoa21cd']].drop_duplicates()
lsoa_msoa.columns = ['LSOA21CD', 'MSOA21CD']

In [11]:
poi_msoa = poi_mapped.merge(lsoa_msoa, on='LSOA21CD', how='left')

# Aggregate to MSOA level
poi_msoa_counts = (
    poi_msoa.groupby(['MSOA21CD', 'poi_group'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print(poi_msoa_counts.shape)
print(poi_msoa_counts.head())

(7264, 12)
poi_group   MSOA21CD  ACCOMMODATION  AUTO  BEAUTY_PERSONAL  EDUCATION  \
0          E02000001             62     3              118         12   
1          E02000002              0     0                4          0   
2          E02000003              0     3               20          1   
3          E02000004              0     0                7          0   
4          E02000005              1     3                4          0   

poi_group  FOOD_DRINK  HEALTH  LEISURE_SPORT  NATURE_OUTDOOR  PROFESSIONAL  \
0                 480     131            164               8          1045   
1                   5       2              2               0             0   
2                  12       8              1               0            27   
3                   1       0              6               0             1   
4                   9       2              2               0             3   

poi_group  RELIGION  RETAIL  
0                55     104  
1                 1  

In [12]:
locomizer_total_od.head()

,o_msoa,d_msoa,flow
0,E02000001,E02000001,22293
1,E02000001,E02000002,8
2,E02000001,E02000003,25
3,E02000001,E02000004,16
4,E02000001,E02000005,28


In [13]:
#remove self-loops
trajectory_od = locomizer_total_od[locomizer_total_od['o_msoa'] != locomizer_total_od['d_msoa']].copy()
trajectory_od['flow_log'] = np.log1p(trajectory_od['flow'])

print(f"After removing self-loops: {len(trajectory_od)}")
print(trajectory_od.head())

After removing self-loops: 2495001
      o_msoa     d_msoa  flow  flow_log
1  E02000001  E02000002     8  2.197225
2  E02000001  E02000003    25  3.258097
3  E02000001  E02000004    16  2.833213
4  E02000001  E02000005    28  3.367296
5  E02000001  E02000007    15  2.772589


In [14]:
# ── Step 2: Node index ────────────────────────────────────────────────────────
all_msoas   = pd.Index(pd.concat([trajectory_od['o_msoa'],
                                   trajectory_od['d_msoa']]).unique())
msoa_to_idx = {msoa: i for i, msoa in enumerate(all_msoas)}

trajectory_od['src'] = trajectory_od['o_msoa'].map(msoa_to_idx)
trajectory_od['dst'] = trajectory_od['d_msoa'].map(msoa_to_idx)

print(f"Unique MSOAs: {len(all_msoas)}")
print(f"OD pairs:     {len(trajectory_od)}")

# ── Step 3: POI feature matrix ────────────────────────────────────────────────
feature_df = pd.DataFrame({'MSOA21CD': list(all_msoas)})
feature_df = feature_df.merge(poi_msoa_counts, on='MSOA21CD', how='left').fillna(0)

assert list(feature_df['MSOA21CD']) == list(all_msoas), "Node order mismatch!"

poi_cols       = [c for c in feature_df.columns if c != 'MSOA21CD']
poi_raw_counts = feature_df[poi_cols].values.copy()

print(f"\nFeature matrix shape:    {feature_df.shape}")
print(f"MSOAs with zero POIs:    {(feature_df[poi_cols].sum(axis=1) == 0).sum()}")
print(f"POI categories:          {poi_cols}")

Unique MSOAs: 7264
OD pairs:     2495001

Feature matrix shape:    (7264, 12)
MSOAs with zero POIs:    0
POI categories:          ['ACCOMMODATION', 'AUTO', 'BEAUTY_PERSONAL', 'EDUCATION', 'FOOD_DRINK', 'HEALTH', 'LEISURE_SPORT', 'NATURE_OUTDOOR', 'PROFESSIONAL', 'RELIGION', 'RETAIL']


In [15]:
#  Load IMD data (LSOA level)
imd_df = pd.read_csv('../data/LSOA_IMD2025_OSGB1936_-7580747902701771840.csv')  

# Load LSOA population data
pop_df= pd.read_csv('../data/new_population_data.csv')

In [16]:
# ── Aggregate population to MSOA ──────────────────────────────────────────────
pop_df['Total'] = pop_df['Total'].astype(str).str.replace(',', '').astype(float)

pop_msoa = (
    pop_df.rename(columns={'LSOA 2021 Code': 'LSOA21CD', 'Total': 'population'})
    .merge(lsoa_msoa, on='LSOA21CD', how='left')
    .groupby('MSOA21CD')['population']
    .sum()
    .reset_index()
)

# ── Aggregate IMD to MSOA (mean decile) ───────────────────────────────────────
imd_msoa = (
    imd_df.rename(columns={'LSOA Code': 'LSOA21CD', 'IMD Decile': 'imd_decile'})
    .merge(lsoa_msoa, on='LSOA21CD', how='left')
    .groupby('MSOA21CD')['imd_decile']
    .mean()
    .round()
    .astype(int)
    .reset_index()
)

# ── Align to node order ───────────────────────────────────────────────────────
eval_df = pd.DataFrame({'MSOA21CD': list(all_msoas)})
eval_df = eval_df.merge(pop_msoa, on='MSOA21CD', how='left')
eval_df = eval_df.merge(imd_msoa, on='MSOA21CD', how='left')
eval_df = eval_df.fillna(0)

population = eval_df['population'].values
imd_decile = eval_df['imd_decile'].values.astype(int)

print(f"Population matched: {(population > 0).sum()} / {len(population)}")
print(f"IMD matched:        {(imd_decile > 0).sum()} / {len(imd_decile)}")
print(f"Population range:   {population.min():.0f} - {population.max():.0f}")
print(f"IMD decile range:   {imd_decile.min()} - {imd_decile.max()}")

Population matched: 7264 / 7264
IMD matched:        6856 / 7264
Population range:   2271 - 17898
IMD decile range:   0 - 10


In [17]:
# Replace 0 IMD with NaN — these are non-England MSOAs
imd_decile_clean = imd_decile.copy().astype(float)
imd_decile_clean[imd_decile_clean == 0] = np.nan

print(f"England MSOAs (IMD available): {(~np.isnan(imd_decile_clean)).sum()}")
print(f"Non-England MSOAs (no IMD):    {np.isnan(imd_decile_clean).sum()}")

# For IMD prediction use only England MSOAs
england_mask = ~np.isnan(imd_decile_clean)
imd_england  = imd_decile_clean[england_mask].astype(int)
poi_england  = poi_raw_counts[england_mask]
emb_england  = None  # will be set after DGI training

print(f"\nEngland MSOAs for IMD evaluation: {england_mask.sum()}")

England MSOAs (IMD available): 6856
Non-England MSOAs (no IMD):    408

England MSOAs for IMD evaluation: 6856


## gravity model

In [18]:
from sklearn.metrics import r2_score, accuracy_score, f1_score, mean_squared_error

In [19]:
# Population
X_train, X_test, y_train, y_test = train_test_split(
    poi_raw_counts, population, test_size=0.2, random_state=42)
grav_pop_r2 = r2_score(y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test))
print(f"\nGravity; Population R²: {grav_pop_r2:.4f}")

# IMD
X_train, X_test, y_train, y_test = train_test_split(
    poi_raw_counts, imd_decile, test_size=0.2, random_state=42)
rf_grav       = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
grav_imd_pred = rf_grav.predict(X_test)
grav_imd_acc  = accuracy_score(y_test, grav_imd_pred)
grav_imd_f1   = f1_score(y_test, grav_imd_pred, average='weighted')
print(f"Gravity → IMD Accuracy: {grav_imd_acc:.4f} | F1: {grav_imd_f1:.4f}")


Gravity; Population R²: 0.1305
Gravity → IMD Accuracy: 0.1328 | F1: 0.1268


### ML

In [20]:
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# PyG graph 
x = torch.tensor(feature_df[poi_cols].values, dtype=torch.float)
x = torch.log1p(x)
x = (x - x.mean(dim=0)) / (x.std(dim=0) + 1e-8)

edge_index  = torch.tensor(trajectory_od[['src', 'dst']].values.T, dtype=torch.long)
edge_weight = torch.tensor(trajectory_od['flow_log'].values, dtype=torch.float)
data        = Data(x=x, edge_index=edge_index, edge_weight=edge_weight)

print(f"Nodes:         {data.num_nodes}")
print(f"Edges:         {data.num_edges}")
print(f"Node features: {data.x.shape}")
print(f"x range:       [{data.x.min():.3f}, {data.x.max():.3f}]")
print(f"Any NaN:       {torch.isnan(data.x).any()}")

Nodes:         7264
Edges:         2495001
Node features: torch.Size([7264, 11])
x range:       [-3.123, 6.817]
Any NaN:       False


In [22]:
# Encoder 
class DGIEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.prelu = torch.nn.PReLU()

    def forward(self, x, edge_index, edge_weight=None):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)
        x = self.prelu(x)
        return x

def corruption(x, edge_index, edge_weight=None):
    return x[torch.randperm(x.size(0))], edge_index, edge_weight

encoder = DGIEncoder(in_channels=data.x.shape[1],
                     hidden_channels=128, out_channels=64)

dgi = DeepGraphInfomax(
    hidden_channels=64,
    encoder=encoder,
    summary=lambda z, *args, **kwargs: torch.sigmoid(z.mean(dim=0)),
    corruption=corruption
)

optimizer = torch.optim.Adam(dgi.parameters(), lr=0.001)

# Training 
dgi.train()
for epoch in range(300):
    optimizer.zero_grad()
    pos_z, neg_z, summary = dgi(data.x, data.edge_index, data.edge_weight)
    loss = dgi.loss(pos_z, neg_z, summary)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

# Extract embeddings    
dgi.eval()
with torch.no_grad():
    embeddings = encoder(data.x, data.edge_index, data.edge_weight)

emb = embeddings.numpy()
print(f"\nEmbeddings: {emb.shape}")
print(f"Range:      [{emb.min():.3f}, {emb.max():.3f}]")
print(f"Std:        {emb.std():.4f}")

Epoch   0 | Loss: 1.3792
Epoch  50 | Loss: 0.0617
Epoch 100 | Loss: 0.0200
Epoch 150 | Loss: 0.0276
Epoch 200 | Loss: 0.0116
Epoch 250 | Loss: 0.0050

Embeddings: (7264, 64)
Range:      [-0.293, 1.564]
Std:        0.1341


In [23]:
from sklearn.metrics import r2_score, accuracy_score, f1_score, mean_squared_error

In [24]:
# Population
X_train, X_test, y_train, y_test = train_test_split(
    emb, population, test_size=0.2, random_state=42)
dgi_pop_r2 = r2_score(y_test,
    LinearRegression().fit(X_train, y_train).predict(X_test))
print(f"DGI → Population R²: {dgi_pop_r2:.4f}")

# IMD
X_train, X_test, y_train, y_test = train_test_split(
    emb, imd_decile, test_size=0.2, random_state=42)
rf_dgi       = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
dgi_imd_pred = rf_dgi.predict(X_test)
dgi_imd_acc  = accuracy_score(y_test, dgi_imd_pred)
dgi_imd_f1   = f1_score(y_test, dgi_imd_pred, average='weighted')
print(f"DGI     → IMD Accuracy: {dgi_imd_acc:.4f} | F1: {dgi_imd_f1:.4f}")

DGI → Population R²: 0.1837
DGI     → IMD Accuracy: 0.2333 | F1: 0.2321


### flow prediction

In [25]:
lsoa_centroids = pd.read_csv('../data/LSOA_PopCentroids_EW_2021_V4_-1224685956027462334.csv')

In [26]:

# ── Aggregate LSOA centroids to MSOA ─────────────────────────────────────────
centroids_msoa = (
    lsoa_centroids.rename(columns={'LSOA21CD': 'LSOA21CD'})
    .merge(lsoa_msoa, on='LSOA21CD', how='left')
    .groupby('MSOA21CD')[['x', 'y']]
    .mean()
    .reset_index()
)

print(f"MSOAs with centroids: {len(centroids_msoa)}")
print(centroids_msoa.head())

MSOAs with centroids: 7264
    MSOA21CD              x              y
0  E02000001  532495.502133  181530.675217
1  E02000002  547975.665950  189412.580550
2  E02000003  548392.525033  188091.886350
3  E02000004  550889.359925  186801.665675
4  E02000005  548635.584917  186872.150250


In [27]:
coord_df = pd.DataFrame({'MSOA21CD': list(all_msoas)})
coord_df = coord_df.merge(centroids_msoa[['MSOA21CD', 'x', 'y']],
                           on='MSOA21CD', how='left')

coords  = torch.tensor(coord_df[['x', 'y']].values, dtype=torch.float)
src_idx = torch.tensor(trajectory_od['src'].values, dtype=torch.long)
dst_idx = torch.tensor(trajectory_od['dst'].values, dtype=torch.long)

dist     = torch.sqrt(((coords[src_idx] - coords[dst_idx]) ** 2).sum(dim=1))
dist_log = torch.log1p(dist)
dist_log = (dist_log - dist_log.mean()) / (dist_log.std() + 1e-8)

od_features = torch.cat([
    embeddings[src_idx],
    embeddings[dst_idx],
    dist_log.unsqueeze(1)
], dim=1).numpy()

flow_target = trajectory_od['flow_log'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    od_features, flow_target, test_size=0.2, random_state=42)

flow_r2     = r2_score(y_te,
    LinearRegression().fit(X_tr, y_tr).predict(X_te))


print(f"\nDGI + Distance → Flow R²:  {flow_r2:.4f}")




DGI + Distance → Flow R²:  0.5691


In [28]:
results_all = pd.DataFrame([
    ("Population",      "Gravity Model (POI)", "R²",       grav_pop_r2),
    ("Population",      "DGI Emb + POI",      "R²",       dgi_pop_r2    ),
    ("IMD Decile",      "Gravity Model (POI)", "Accuracy", grav_imd_acc ),
    ("IMD Decile",      "DGI Emb + POI",      "Accuracy", dgi_imd_acc   ),
    ("Flow Prediction", "DGI + Distance",      "R²",       flow_r2      ),
], columns=["Task", "Model", "Metric", "Trajectory_data"])

print(results_all.to_string(index=False))


           Task               Model   Metric  Trajectory_data
     Population Gravity Model (POI)       R²         0.130461
     Population       DGI Emb + POI       R²         0.183734
     IMD Decile Gravity Model (POI) Accuracy         0.132829
     IMD Decile       DGI Emb + POI Accuracy         0.233310
Flow Prediction      DGI + Distance       R²         0.569140


In [29]:
# Leeds — swap variables
pop_series     = pd.Series(population, index=list(all_msoas))
origin_pop     = pop_series[trajectory_od['o_msoa']].values
dest_pop       = pop_series[trajectory_od['d_msoa']].values
log_origin_pop = np.log1p(origin_pop)
log_dest_pop   = np.log1p(dest_pop)

gravity_features = np.column_stack([
    log_origin_pop,
    log_dest_pop,
    dist_log.numpy()
])

flow_target  = trajectory_od['flow_log'].values
X_tr, X_te, y_tr, y_te = train_test_split(
    gravity_features, flow_target, test_size=0.2, random_state=42)

grav_flow_r2_traj = r2_score(y_te,
    LinearRegression().fit(X_tr, y_tr).predict(X_te))
print(f"Gravity Model → Flow R²: {grav_flow_r2_traj:.4f}")

Gravity Model → Flow R²: 0.4053


In [30]:
results_all = pd.DataFrame([
    ("Population",      "Gravity Model (POI)", "R²",       grav_pop_r2),
    ("Population",      "DGI Emb + POI",      "R²",       dgi_pop_r2    ),
    ("IMD Decile",      "Gravity Model (POI)", "Accuracy", grav_imd_acc ),
    ("IMD Decile",      "DGI Emb + POI",      "Accuracy", dgi_imd_acc   ),
    ("Flow Prediction", "DGI + Distance",      "R²",       flow_r2      ),
    ("Flow Prediction", "Population + Distance",    "R²", grav_flow_r2_traj ), 

], columns=["Task", "Model", "Metric", "London"])

print(results_all.to_string(index=False))

           Task                 Model   Metric   London
     Population   Gravity Model (POI)       R² 0.130461
     Population         DGI Emb + POI       R² 0.183734
     IMD Decile   Gravity Model (POI) Accuracy 0.132829
     IMD Decile         DGI Emb + POI Accuracy 0.233310
Flow Prediction        DGI + Distance       R² 0.569140
Flow Prediction Population + Distance       R² 0.405338
